# 04 — RAG Avançado com ChromaDB + **LangChain 1.x (LCEL)**

**Módulo:** EAI_07 — IA Generativa  
**Submódulo:** 03_RAG  
**Ambiente:** `eai07` (Python 3.11)

---

## Importante — versão do LangChain

A partir do LangChain **1.0**, os módulos antigos foram removidos:

| Removido (0.x) | Substituto (1.x) |
|---|---|
| `langchain.memory` | `langchain_community.chat_message_histories.ChatMessageHistory` |
| `langchain.chains.ConversationalRetrievalChain` | LCEL: `chain = prompt \| llm \| parser` |
| `langchain.prompts` | `langchain_core.prompts` |
| `langchain.schema` | `langchain_core.messages` |

> O banco ChromaDB em disco **não muda** — só o código Python que acessa ele.

## Instalação (executar uma vez)

```bash
conda activate eai07
pip install langchain langchain-community langchain-chroma langchain-openai langchain-huggingface
```

## 1. Setup completo — imports, clientes e vectorstore

In [17]:
import sys, os, re, time
from dotenv import load_dotenv

# ── LangChain 1.x — imports corretos ──────────────────────────────────────
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

sys.path.append(os.path.abspath('..'))
load_dotenv('../.env')

# ── Embeddings ────────────────────────────────────────────────────────────
embedding_function = HuggingFaceEmbeddings(
    model_name='all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True},
)

# ── LLM — DeepSeek via interface OpenAI-compatível ────────────────────────
llm = ChatOpenAI(
    model=os.getenv('LLM_MODEL', 'deepseek-chat'),
    api_key=os.getenv('DEEPSEEK_API_KEY'),
    base_url='https://api.deepseek.com',
    temperature=0.2,
    max_tokens=600,
)

# ── Vectorstore — abre (ou cria) o banco ChromaDB em disco ────────────────
CHROMA_PATH     = '../data/chroma_db'
COLLECTION_NAME = 'agent_contexts'
PROJETO_BASE    = os.path.abspath('../..')

os.makedirs(CHROMA_PATH, exist_ok=True)

vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_function,
    persist_directory=CHROMA_PATH,
    collection_metadata={'hnsw:space': 'cosine'},
)

retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5},
)

print(f'LLM       : {llm.model_name}')
print(f'Embedding : {embedding_function.model_name}')
print(f'Projeto   : {PROJETO_BASE}')
print(f'Chunks    : {vectorstore._collection.count()}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


LLM       : deepseek-chat
Embedding : all-MiniLM-L6-v2
Projeto   : C:\Users\pcwin\Documents\Especialista_em_AI
Chunks    : 1763


## 2. Funções de processamento de chunks

In [18]:
ENRIQUECIMENTO = {
    'regressão linear'       : 'regressão linear, ajuste de curva, reta, mínimos quadrados, ajustar linha, coeficientes a e b',
    'mínimos quadrados'      : 'mínimos quadrados, regressão linear, ajuste de reta, least squares',
    'MSE'                    : 'MSE, Mean Squared Error, erro quadrático médio',
    'CNN'                    : 'CNN, rede convolucional, convolutional neural network, redes convolucionais, conv2D',
    'LSTM'                   : 'LSTM, Long Short-Term Memory, células de memória, gates, sequências',
    'deep learning'          : 'deep learning, aprendizado profundo, redes neurais profundas, DL',
    'ANN'                    : 'ANN, rede neural artificial, perceptron, MLP',
    'GRU'                    : 'GRU, Gated Recurrent Unit, células recorrentes',
    'KNN'                    : 'KNN, K-Nearest Neighbors, vizinhos mais próximos',
    'Random Forest'          : 'Random Forest, floresta aleatória, ensemble, árvores de decisão',
    'SVM'                    : 'SVM, Support Vector Machine',
    'embeddings'             : 'embeddings, word embeddings, vetores de palavras, representação vetorial',
    'TF-IDF'                 : 'TF-IDF, term frequency, bag of words, BoW',
    'transformers'           : 'transformers, BERT, GPT, attention, mecanismo de atenção',
    'autovalores'            : 'autovalores, autovetores, eigenvalues, PCA',
    'transformações lineares': 'transformações lineares, matrizes, rotação, escala, cisalhamento',
    'OpenCV'                 : 'OpenCV, visão computacional, processamento de imagens',
    'YOLO'                   : 'YOLO, YOLOv5, detecção de objetos, object detection',
    'reconhecimento facial'  : 'reconhecimento facial, face recognition, detecção de rostos',
    'RAG'                    : 'RAG, Retrieval Augmented Generation, recuperação de documentos, busca semântica',
    'function calling'       : 'function calling, tool calling, ferramentas, tools, agentes, modelo decide',
    'prompt engineering'     : 'prompt engineering, zero-shot, few-shot, chain-of-thought, CoT',
    'MLflow'                 : 'MLflow, rastreamento de experimentos, experiment tracking',
    'drift'                  : 'drift, data drift, monitoramento, degradação do modelo',
}

def enriquecer_chunk(texto, modulo='', arquivo=''):
    prefixo = f'[{modulo}' + (f' / {arquivo}' if arquivo else '') + '] ' if modulo else ''
    sinonimos = [v for k, v in ENRIQUECIMENTO.items() if k.lower() in texto.lower()]
    resultado = prefixo + texto
    if sinonimos:
        resultado += ' | ' + '; '.join(sinonimos)
    return resultado

def chunk_por_secao(texto):
    chunks, titulo, linhas = [], 'Introdução', []
    for linha in texto.split('\n'):
        if linha.startswith('#'):
            if linhas:
                c = ' '.join(linhas).strip()
                if c: chunks.append({'titulo': titulo, 'conteudo': c})
            titulo, linhas = linha.lstrip('#').strip(), []
        elif linha.strip():
            linhas.append(linha.strip())
    if linhas:
        c = ' '.join(linhas).strip()
        if c: chunks.append({'titulo': titulo, 'conteudo': c})
    return chunks

def processar_agent_context(conteudo, modulo, arquivo=''):
    import re as _re
    m = _re.match(r'(EAI_\d+)', modulo)
    modulo_prefixo = m.group(1) if m else modulo
    chunks = []
    for s in chunk_por_secao(conteudo):
        resumo = ' '.join(s['conteudo'].split()[:40])
        chunks.append({
            'chunk_busca'    : enriquecer_chunk(f"{s['titulo']}: {resumo}", modulo=modulo, arquivo=arquivo),
            'chunk_contexto' : f"[{modulo} — {s['titulo']}]\n{s['conteudo']}",
            'titulo'         : s['titulo'],
            'modulo'         : modulo,
            'modulo_prefixo' : modulo_prefixo,
            'arquivo'        : arquivo,
        })
    return chunks

def encontrar_agent_contexts(pasta_raiz):
    encontrados, ignorar = [], {'.git', 'venv', '.venv', '__pycache__', 'node_modules'}
    for raiz, dirs, arquivos in os.walk(pasta_raiz):
        dirs[:] = [d for d in dirs if d not in ignorar and not d.startswith('.')]
        if 'AGENT_CONTEXT.md' in arquivos:
            partes = raiz.replace('\\', '/').split('/')
            modulo = next((p for p in partes if p.startswith('EAI_')), os.path.basename(raiz))
            caminho_arquivo = os.path.join(raiz, 'AGENT_CONTEXT.md')
            caminho_rel = os.path.relpath(caminho_arquivo, pasta_raiz).replace('\\', '/')
            encontrados.append((modulo, caminho_arquivo, caminho_rel))
    return sorted(encontrados)

print('Funções de processamento carregadas.')

Funções de processamento carregadas.


## 3. Indexação (executa só se o banco estiver vazio)

In [19]:
def indexar_projeto(pasta_raiz: str, force: bool = False):
    total_existente = vectorstore._collection.count()
    if total_existente > 0 and not force:
        print(f'Banco já populado ({total_existente} chunks). Pulando indexação.')
        print('Use indexar_projeto(..., force=True) para forçar reindexação.')
        return

    arquivos = encontrar_agent_contexts(pasta_raiz)
    print(f'Encontrados {len(arquivos)} AGENT_CONTEXT.md em {pasta_raiz}')
    total_chunks, t0 = 0, time.time()

    for modulo, caminho_abs, caminho_rel in arquivos:
        with open(caminho_abs, 'r', encoding='utf-8') as f:
            conteudo = f.read()
        chunks = processar_agent_context(conteudo, modulo, arquivo=caminho_rel)
        if not chunks:
            continue
        arquivo_slug = caminho_rel.replace('/', '__').replace('\\', '__').replace('.', '_')
        vectorstore.add_texts(
            texts     = [c['chunk_busca'] for c in chunks],
            metadatas = [
                {
                    'modulo'        : c['modulo'],
                    'modulo_prefixo': c['modulo_prefixo'],
                    'titulo'        : c['titulo'],
                    'arquivo'       : c['arquivo'],
                    '_contexto'     : c['chunk_contexto'],
                }
                for c in chunks
            ],
            ids = [f"{arquivo_slug}__chunk_{i:04d}" for i in range(len(chunks))],
        )
        total_chunks += len(chunks)
        print(f'  [{modulo}] {len(chunks)} chunks indexados')

    print(f'\nIndexação concluída: {total_chunks} chunks em {time.time()-t0:.1f}s')
    print(f'Total no banco: {vectorstore._collection.count()} chunks')


indexar_projeto(PROJETO_BASE)

Banco já populado (1763 chunks). Pulando indexação.
Use indexar_projeto(..., force=True) para forçar reindexação.


## 4. Busca semântica

In [20]:
def buscar(
    query: str,
    top_k: int = 5,
    score_minimo: float = 0.3,
    filtro_modulo: str = None,
) -> list:
    kwargs = {'k': top_k}
    if filtro_modulo:
        kwargs['filter'] = {'modulo_prefixo': {'$eq': filtro_modulo}}

    pares = vectorstore.similarity_search_with_score(query, **kwargs)

    resultados = []
    for doc, dist in pares:
        score = 1.0 - (dist / 2.0)
        if score >= score_minimo:
            resultados.append({
                'contexto': doc.metadata.get('_contexto', doc.page_content),
                'score'   : score,
                'meta'    : {k: v for k, v in doc.metadata.items() if k != '_contexto'},
            })
    return resultados


# ── Teste ─────────────────────────────────────────────────────────────────
query = 'como foi implementada a regressão linear?'

print('── Sem filtro ───────────────────────────────────────────')
for r in buscar(query, top_k=3):
    print(f"  [{r['score']:.3f}] {r['meta']}")

print()
print('── Com filtro EAI_01 ────────────────────────────────────')
for r in buscar(query, top_k=3, filtro_modulo='EAI_01'):
    print(f"  [{r['score']:.3f}] {r['meta']}")

── Sem filtro ───────────────────────────────────────────
  [0.766] {'arquivo': 'EAI_07_AI_Generative/03_RAG/AGENT_CONTEXT.md', 'modulo_prefixo': 'EAI_07', 'titulo': 'Query Expansion', 'modulo': 'EAI_07_AI_Generative'}
  [0.764] {'titulo': 'Classificação', 'arquivo': 'EAI_02_Machine_Learning/AGENT_CONTEXT.md', 'modulo_prefixo': 'EAI_02', 'modulo': 'EAI_02_Machine_Learning'}
  [0.763] {'modulo_prefixo': 'EAI_01', 'modulo': 'EAI_01_Fundamentos_Matemática_para_IA', 'arquivo': 'EAI_01_Fundamentos_Matemática_para_IA/AGENT_CONTEXT.md', 'titulo': '5. regressao_manual.ipynb'}

── Com filtro EAI_01 ────────────────────────────────────
  [0.763] {'modulo_prefixo': 'EAI_01', 'arquivo': 'EAI_01_Fundamentos_Matemática_para_IA/AGENT_CONTEXT.md', 'titulo': '5. regressao_manual.ipynb', 'modulo': 'EAI_01_Fundamentos_Matemática_para_IA'}
  [0.741] {'modulo_prefixo': 'EAI_01', 'titulo': 'Regressão linear manual', 'modulo': 'EAI_01_Fundamentos_Matemática_para_IA', 'arquivo': 'EAI_01_Fundamentos_Matemática

## 5. Query Expansion

In [21]:
EXPANSION_TEMPLATE = PromptTemplate.from_template(
    """Você é um especialista em IA. Reformule a pergunta abaixo em termos técnicos \
mais precisos para melhorar uma busca semântica em documentação técnica de IA.

REGRAS:
1. Se a pergunta for INDEPENDENTE, ignore o histórico e expanda com sinônimos técnicos.
2. Se for CONTINUAÇÃO (usa 'desse projeto', 'nele', 'qual foi a acurácia' etc.),
   use o histórico para resolver a referência e inclua os termos concretos.
3. Responda APENAS com a query reformulada. Máximo de 2 linhas.{contexto_historico}

Pergunta original: {pergunta}
Query reformulada:"""
)

# LCEL: chain = prompt | llm | parser
expansion_chain = EXPANSION_TEMPLATE | llm | StrOutputParser()

def expandir_query(pergunta: str, historico: list = None) -> str:
    """
    historico: lista de dicts {'role': ..., 'content': ...}
    """
    if historico:
        ultimas = historico[-4:]
        ctx_linhas = [
            ('Usuario' if m['role'] == 'user' else 'Assistente') + f": {m['content'][:300]}"
            for m in ultimas
        ]
        contexto_str = '\n\nHistórico recente:\n' + '\n'.join(ctx_linhas) + '\n'
    else:
        contexto_str = ''

    # .invoke() recebe o dict com as variáveis do template
    return expansion_chain.invoke({
        'contexto_historico': contexto_str,
        'pergunta': pergunta,
    })


# ── Teste ─────────────────────────────────────────────────────────────────
print('Query Expansion:\n')
for p in [
    'como ajustar uma linha aos pontos de dados?',
    'como o modelo decide qual ferramenta chamar?',
]:
    print(f'  Original  : {p}')
    print(f'  Expandida : {expandir_query(p)}')
    print()

Query Expansion:

  Original  : como ajustar uma linha aos pontos de dados?
  Expandida : Técnicas de ajuste de curva e regressão para modelagem de dados: métodos de otimização de parâmetros (mínimos quadrados, regressão linear/polinomial) para minimizar erro residual entre modelo e pontos de dados observados.

  Original  : como o modelo decide qual ferramenta chamar?
  Expandida : Mecanismo de seleção de ferramentas em modelos de linguagem: processo de decisão baseado em embeddings de contexto, similaridade semântica e classificação de intenção para invocação de API ou função externa.



## 6. Reranking

In [22]:
RERANK_TEMPLATE = PromptTemplate.from_template(
    """Pergunta: {query}

Avalie os chunks abaixo por relevância para responder a pergunta.
Responda APENAS com os números em ordem de relevância (ex: 3,1,4,2).
Não inclua explicações.

Chunks:
{chunks_texto}

Ordem por relevância:"""
)

rerank_chain = RERANK_TEMPLATE | llm | StrOutputParser()

def rerankar(query: str, resultados: list) -> list:
    if len(resultados) <= 1:
        return resultados

    chunks_texto = '\n\n'.join(
        f"[{i+1}] {r['meta']}\n{r['contexto'][:300]}..."
        for i, r in enumerate(resultados)
    )
    ordem_str = rerank_chain.invoke({'query': query, 'chunks_texto': chunks_texto})

    try:
        numeros = [int(x) for x in re.findall(r'\d+', ordem_str)]
        numeros = [n for n in numeros if 1 <= n <= len(resultados)]
        faltando = [n for n in range(1, len(resultados)+1) if n not in numeros]
        return [resultados[i-1] for i in (numeros + faltando)]
    except Exception:
        return resultados


# ── Teste ─────────────────────────────────────────────────────────────────
query = 'como foi implementada a regressão linear no projeto?'
resultados = buscar(query, top_k=4)

print('── Antes do reranking ───────────────────────────────────')
for i, r in enumerate(resultados):
    print(f"  [{i+1}] score={r['score']:.3f} {r['meta']}")

print()
print('── Depois do reranking ──────────────────────────────────')
for i, r in enumerate(rerankar(query, resultados)):
    print(f"  [{i+1}] score={r['score']:.3f} {r['meta']}")

── Antes do reranking ───────────────────────────────────
  [1] score=0.773 {'modulo_prefixo': 'EAI_04', 'modulo': 'EAI_04_NLP_Classico', 'titulo': 'Progressão de Complexidade', 'arquivo': 'EAI_04_NLP_Classico/AGENT_CONTEXT.md'}
  [2] score=0.765 {'titulo': 'Classificação', 'arquivo': 'EAI_02_Machine_Learning/AGENT_CONTEXT.md', 'modulo': 'EAI_02_Machine_Learning', 'modulo_prefixo': 'EAI_02'}
  [3] score=0.762 {'modulo_prefixo': 'EAI_04', 'arquivo': 'EAI_04_NLP_Classico/AGENT_CONTEXT.md', 'titulo': 'Estrutura Pedagógica', 'modulo': 'EAI_04_NLP_Classico'}
  [4] score=0.760 {'arquivo': 'EAI_01_Fundamentos_Matemática_para_IA/AGENT_CONTEXT.md', 'modulo_prefixo': 'EAI_01', 'modulo': 'EAI_01_Fundamentos_Matemática_para_IA', 'titulo': 'RESUMO EXECUTIVO'}

── Depois do reranking ──────────────────────────────────
  [1] score=0.760 {'arquivo': 'EAI_01_Fundamentos_Matemática_para_IA/AGENT_CONTEXT.md', 'modulo_prefixo': 'EAI_01', 'modulo': 'EAI_01_Fundamentos_Matemática_para_IA', 'titulo': 'RESUMO

## 7. Pipeline RAG — Opção A: LCEL com `RunnableWithMessageHistory`

Substituto moderno do `ConversationalRetrievalChain`.  
`ChatMessageHistory` armazena o histórico; `RunnableWithMessageHistory` injeta automaticamente.

In [23]:
SYSTEM_PROMPT_STR = """Você é o Assistente Técnico do projeto ESPECIALISTA_EM_IA de Carlos Henrique.
Módulos: EAI_00 (Install Config), EAI_01 (Fundamentos Matemáticos), EAI_02 (Machine Learning),
EAI_03 (Deep Learning), EAI_04 (NLP Clássico), EAI_05 (NLP Transformers),
EAI_06 (Visão Computacional), EAI_07 (IA Generativa), EAI_08 (MLOps).
Responda em português. Use o contexto fornecido. Seja direto e técnico."""

# Prompt com placeholder para o histórico de mensagens
rag_prompt = ChatPromptTemplate.from_messages([
    ('system', SYSTEM_PROMPT_STR + '\n\nContexto:\n{contexto}'),
    MessagesPlaceholder(variable_name='historico'),
    ('human', '{pergunta}'),
])

# Chain LCEL: prompt → llm → parser
rag_chain_base = rag_prompt | llm | StrOutputParser()

# Armazena um histórico por session_id
historicos: dict[str, ChatMessageHistory] = {}

def get_historico(session_id: str) -> ChatMessageHistory:
    if session_id not in historicos:
        historicos[session_id] = ChatMessageHistory()
    return historicos[session_id]

# Envolve a chain com gerenciamento automático de histórico
rag_chain_com_historico = RunnableWithMessageHistory(
    rag_chain_base,
    get_historico,
    input_messages_key='pergunta',
    history_messages_key='historico',
)

# Histórico bruto por session_id para usar no expandir_query
_historicos_raw: dict[str, list] = {}

def responder_simples(
    pergunta: str,
    session_id: str = 'default',
    top_k: int = 5,
    filtro_modulo: str = None,
    usar_expansion: bool = True,
    usar_reranking: bool = True,
    verbose: bool = False,
) -> str:
    """
    Pipeline LCEL com query expansion + reranking + histórico automático.
    Equivalente à Opção B, mas sem encapsular em classe.
    """
    # Histórico bruto desta sessão (para expandir_query resolver referências)
    hist_raw = _historicos_raw.setdefault(session_id, [])

    # 1. Query expansion com contexto do histórico
    query_busca = expandir_query(pergunta, historico=hist_raw[-4:]) \
                  if usar_expansion else pergunta
    if verbose:
        print(f'Query busca: {query_busca[:80]}')

    # 2. Busca semântica
    resultados = buscar(query_busca, top_k=top_k, filtro_modulo=filtro_modulo)

    # 3. Reranking
    if usar_reranking and len(resultados) > 1:
        resultados = rerankar(pergunta, resultados)

    if verbose:
        print(f'Chunks: {len(resultados)}')
        for r in resultados:
            print(f"  [{r['score']:.3f}] {r['meta']}")

    # 4. Monta contexto
    contexto = '\n\n---\n\n'.join(r['contexto'] for r in resultados[:3]) \
               if resultados else 'Sem contexto relevante encontrado.'

    # 5. Invoca chain LCEL (histórico injetado automaticamente pelo RunnableWithMessageHistory)
    resposta = rag_chain_com_historico.invoke(
        {'pergunta': pergunta, 'contexto': contexto},
        config={'configurable': {'session_id': session_id}},
    )

    # 6. Salva histórico bruto para próximas expansões
    hist_raw.append({'role': 'user',      'content': pergunta})
    hist_raw.append({'role': 'assistant', 'content': resposta})

    return resposta


print('Pipeline LCEL (Opção A) pronto.')

# ── Teste ─────────────────────────────────────────────────────────────────
for pergunta in [
    'Qual projeto de deep learning classificou obras de arte?',
    'Qual foi a acurácia desse projeto?',
    'Quais técnicas de aumento de dados foram usadas nele?',
]:
    print(f'\n👤 {pergunta}')
    print('─' * 55)
    print(f'🤖 {responder_simples(pergunta, session_id="sessao_a")}')

Pipeline LCEL (Opção A) pronto.

👤 Qual projeto de deep learning classificou obras de arte?
───────────────────────────────────────────────────────
🤖 O projeto que classificou obras de arte foi o **EAI_03_Deep_Learning — Projeto de Classificação de Pinturas por Estilo Artístico**, que utilizou **Transfer Learning** com a arquitetura **MobileNetV2** (pré-treinada no ImageNet) + **Fine-tuning** no dataset **WikiArt - Painter by Numbers** (6-7 estilos). O resultado foi ~65% de accuracy, com deploy em Flask web app para upload de imagens.

👤 Qual foi a acurácia desse projeto?
───────────────────────────────────────────────────────
🤖 A acurácia do projeto foi de **~65%**. Esse valor é considerado competitivo para o problema, dado que a classificação de estilos artísticos é uma tarefa subjetiva e complexa, com sobreposição visual entre estilos e sem contexto histórico disponível para o modelo.

👤 Quais técnicas de aumento de dados foram usadas nele?
──────────────────────────────────────────

## 8. Pipeline RAG — Opção B: `AssistenteRAG` completo

Mantém a mesma interface do notebook anterior com Query Expansion + Reranking + filtro por módulo,
usando os componentes LCEL internamente.

In [24]:
class AssistenteRAG:
    """
    RAG avançado (LangChain 1.x / LCEL):
    - Query Expansion via chain LCEL (PromptTemplate | llm | StrOutputParser)
    - Reranking dos chunks recuperados
    - Histórico via ChatMessageHistory + RunnableWithMessageHistory
    - Filtro opcional por módulo
    """
    def __init__(self, session_id: str = None, max_historico: int = 6):
        self.session_id = session_id or f'rag_{int(time.time())}'
        self.max_historico = max_historico
        self._chat_history = ChatMessageHistory()
        self._historico_raw = []   # formato {'role', 'content'} para expandir_query

        # Chain LCEL com histórico injetado automaticamente
        self._chain = RunnableWithMessageHistory(
            rag_chain_base,
            lambda sid: self._chat_history,
            input_messages_key='pergunta',
            history_messages_key='historico',
        )

    def responder(
        self,
        pergunta: str,
        top_k: int = 5,
        filtro_modulo: str = None,
        usar_expansion: bool = True,
        usar_reranking: bool = True,
        verbose: bool = False,
    ) -> str:

        # 1. Query expansion
        query_busca = expandir_query(pergunta, historico=self._historico_raw[-4:]) \
                      if usar_expansion else pergunta
        if verbose:
            print(f'Query busca: {query_busca[:80]}')

        # 2. Busca semântica
        resultados = buscar(query_busca, top_k=top_k, filtro_modulo=filtro_modulo)

        # 3. Reranking
        if usar_reranking and len(resultados) > 1:
            resultados = rerankar(pergunta, resultados)

        if verbose:
            print(f'Chunks: {len(resultados)}')
            for r in resultados:
                print(f"  [{r['score']:.3f}] {r['meta']}")

        # 4. Monta contexto
        contexto = '\n\n---\n\n'.join(r['contexto'] for r in resultados[:3]) \
                   if resultados else 'Sem contexto relevante encontrado.'

        # 5. Invoca chain LCEL (histórico injetado automaticamente)
        resposta = self._chain.invoke(
            {'pergunta': pergunta, 'contexto': contexto},
            config={'configurable': {'session_id': self.session_id}},
        )

        # 6. Salva no cache bruto para expandir_query
        self._historico_raw.append({'role': 'user',      'content': pergunta})
        self._historico_raw.append({'role': 'assistant', 'content': resposta})

        # Limita histórico ao máximo configurado
        if len(self._chat_history.messages) > self.max_historico * 2:
            self._chat_history.messages = self._chat_history.messages[-(self.max_historico * 2):]

        return resposta

    def limpar_historico(self):
        self._chat_history.clear()
        self._historico_raw = []
        print('Histórico limpo.')


print('AssistenteRAG (LCEL) definido.')

AssistenteRAG (LCEL) definido.


In [25]:
# Teste com perguntas em sequência
assistente = AssistenteRAG()

for pergunta in [
    'Qual projeto de deep learning classificou obras de arte?',
    'Qual foi a acurácia desse projeto?',
    'Quais técnicas de aumento de dados foram usadas nele?',
    'Quais foi o algoritimo utilizado no projeto de reconhecimento facial?',
]:
    print(f'\n👤 {pergunta}')
    print('─' * 55)
    print(f'🤖 {assistente.responder(pergunta)}')


👤 Qual projeto de deep learning classificou obras de arte?
───────────────────────────────────────────────────────
🤖 O projeto que classificou obras de arte foi o **Classificador de Pinturas por Estilo Artístico**, parte do módulo **EAI_03_Deep_Learning**.

**Detalhes técnicos:**
- **Arquitetura**: MobileNetV2 pré-treinada no ImageNet com fine-tuning
- **Dataset**: WikiArt - Painter by Numbers (6-7 estilos artísticos)
- **Resultado**: ~65% de acurácia (considerado satisfatório para a complexidade do problema)
- **Deployment**: Aplicação web Flask com upload de imagens
- **Diferencial**: Inclui análise profunda de erros e capacidade de generalização

**Observação**: Este projeto difere dos outros 4 projetos do módulo (MNIST MLP, CNN, Regressão MLP, Regularização Visual) que focam em problemas clássicos de nível iniciante a intermediário.

👤 Qual foi a acurácia desse projeto?
───────────────────────────────────────────────────────
🤖 A acurácia do projeto foi de **~65%** (aproximadamente

In [26]:
# Teste com filtro por módulo
assistente2 = AssistenteRAG()

resposta = assistente2.responder(
    'Como calcular a reta que melhor se ajusta a dados de altura e peso?',
    filtro_modulo='EAI_01',
    verbose=True,
)
print(f'\n🤖 {resposta}')

Query busca: Métodos de regressão linear para estimar parâmetros de ajuste de curva em dados 
Chunks: 5
  [0.788] {'modulo_prefixo': 'EAI_01', 'modulo': 'EAI_01_Fundamentos_Matemática_para_IA', 'arquivo': 'EAI_01_Fundamentos_Matemática_para_IA/AGENT_CONTEXT.md', 'titulo': 'Regressão Linear'}
  [0.778] {'arquivo': 'EAI_01_Fundamentos_Matemática_para_IA/AGENT_CONTEXT.md', 'modulo': 'EAI_01_Fundamentos_Matemática_para_IA', 'titulo': '5. regressao_manual.ipynb', 'modulo_prefixo': 'EAI_01'}
  [0.764] {'arquivo': 'EAI_01_Fundamentos_Matemática_para_IA/AGENT_CONTEXT.md', 'modulo': 'EAI_01_Fundamentos_Matemática_para_IA', 'modulo_prefixo': 'EAI_01', 'titulo': 'Regressão linear manual'}
  [0.770] {'titulo': 'Previsão', 'arquivo': 'EAI_01_Fundamentos_Matemática_para_IA/AGENT_CONTEXT.md', 'modulo': 'EAI_01_Fundamentos_Matemática_para_IA', 'modulo_prefixo': 'EAI_01'}
  [0.768] {'arquivo': 'EAI_01_Fundamentos_Matemática_para_IA/AGENT_CONTEXT.md', 'modulo': 'EAI_01_Fundamentos_Matemática_para_IA', '

## 9. Utilitários — inspeção e manutenção

In [27]:
def inspecionar_colecao():
    total = vectorstore._collection.count()
    print(f'Total de chunks: {total}\n')
    todos = vectorstore._collection.get(include=['metadatas'])
    modulos = {}
    for meta in todos['metadatas']:
        m = meta.get('modulo', 'desconhecido')
        modulos[m] = modulos.get(m, 0) + 1
    print('Chunks por módulo:')
    for modulo, count in sorted(modulos.items()):
        print(f'  {modulo:<45} {count:>4} chunks')

inspecionar_colecao()

Total de chunks: 1763

Chunks por módulo:
  EAI_00_Install_Config                           18 chunks
  EAI_01_Fundamentos_Matemática_para_IA           26 chunks
  EAI_02_Machine_Learning                        275 chunks
  EAI_03_Deep_Learning                           344 chunks
  EAI_04_NLP_Classico                            413 chunks
  EAI_05_NLP_com_Transformers                    195 chunks
  EAI_06_Visao_Computacional                     225 chunks
  EAI_07_AI_Generative                           210 chunks
  EAI_08_MLOps_e_Implantação                      57 chunks


In [28]:
def atualizar_modulo(pasta_raiz: str, prefixo_modulo: str):
    """Reindexação incremental de um módulo específico."""
    arquivos = [
        (m, c, r) for m, c, r in encontrar_agent_contexts(pasta_raiz)
        if m.startswith(prefixo_modulo)
    ]
    if not arquivos:
        print(f'Nenhum AGENT_CONTEXT.md encontrado para {prefixo_modulo}')
        return
    for modulo, caminho_abs, caminho_rel in arquivos:
        with open(caminho_abs, 'r', encoding='utf-8') as f:
            conteudo = f.read()
        chunks = processar_agent_context(conteudo, modulo, arquivo=caminho_rel)
        arquivo_slug = caminho_rel.replace('/', '__').replace('\\', '__').replace('.', '_')
        vectorstore.add_texts(
            texts     = [c['chunk_busca'] for c in chunks],
            metadatas = [{'modulo': c['modulo'], 'modulo_prefixo': c['modulo_prefixo'],
                          'titulo': c['titulo'], 'arquivo': c['arquivo'],
                          '_contexto': c['chunk_contexto']} for c in chunks],
            ids = [f"{arquivo_slug}__chunk_{i:04d}" for i in range(len(chunks))],
        )
        print(f'Módulo {modulo} atualizado: {len(chunks)} chunks')

# atualizar_modulo(PROJETO_BASE, 'EAI_07')


def resetar_banco(confirmar: bool = False):
    """Apaga e recria o banco. Use confirmar=True para executar."""
    global vectorstore, retriever
    if not confirmar:
        print('Passe confirmar=True para apagar o banco.')
        return
    vectorstore._client.delete_collection(COLLECTION_NAME)
    vectorstore = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embedding_function,
        persist_directory=CHROMA_PATH,
        collection_metadata={'hnsw:space': 'cosine'},
    )
    retriever = vectorstore.as_retriever(search_type='similarity', search_kwargs={'k': 5})
    print('Banco resetado. Reindexando...')
    indexar_projeto(PROJETO_BASE, force=True)

# resetar_banco(confirmar=True)

---
## Resumo — LangChain 0.x vs 1.x

| Componente | LangChain 0.x (removido) | LangChain 1.x (correto) |
|---|---|---|
| Mensagens | `langchain.schema` | `langchain_core.messages` |
| Prompts | `langchain.prompts` | `langchain_core.prompts` |
| Parser | — | `langchain_core.output_parsers.StrOutputParser` |
| Histórico | `langchain.memory.ConversationBufferWindowMemory` | `langchain_community.chat_message_histories.ChatMessageHistory` |
| Pipeline | `ConversationalRetrievalChain` | LCEL: `prompt \| llm \| parser` |
| Histórico automático | `memory=` no chain | `RunnableWithMessageHistory` |

**Kernel → Restart & Run All** executa tudo na ordem correta.